# Auditoría de datos v2: Times.csv y Monitoring.csv
TA Aprendizaje de Máquina, grupo 5. Entregable 1, fase de comprensión de datos.

**Qué cambió respecto a la v1.** La v1 asumía que `start_time`, `end_time` y `difference_time` eran valores simples. En los datos reales son **listas con un valor por punto de control** (`{05:45:00,05:47:00,...}`, 18 elementos). Esta versión las lee como listas y agrega el análisis de puntualidad por punto de control. También usa el dato de clase de que en Monitoring `lap = 0` significa vehículo fuera de operación.

Este notebook describe los datos y responde las verificaciones del plan v2. No elimina filas del archivo original ni entrena modelos: cuenta, compara y grafica. Las dos únicas "decisiones" que toma (regla de valores extremos y partición temporal de referencia) están etiquetadas como `decision_provisional` o `referencia_preliminar`.

Cómo usarlo:

1. Coloca Times.csv y Monitoring.csv en `data/raw/`, o cambia RUTA_TIMES y RUTA_MONITORING en la celda de configuración. En Google Colab, si no los encuentra, se abrirá el selector para subirlos. Si Monitoring.csv es muy pesado, conviene montar Google Drive y poner su ruta.
2. Ejecuta todas las celdas en orden (Entorno de ejecución, Ejecutar todas). Con Monitoring de 1,6 millones de filas tarda unos minutos y usa cerca de 2 GB de memoria.
3. Al final se genera la carpeta `results/auditoria` con resumen_auditoria.md, hallazgos.json y las figuras, y un archivo `results/auditoria.zip` (ambos ignorados por git).

Etiquetas: `observado` es un conteo o estadístico calculado; `indicio` es una interpretación automática que el equipo debe confirmar; `dato_de_clase` viene de los apuntes de clase; `decision_provisional` es una elección del notebook para seguir; `referencia_preliminar` es una cifra de contexto, no un experimento; `error` indica un bloque que no pudo ejecutarse.

## 0. Configuración

In [1]:
from pathlib import Path

RUTA_TIMES = "../data/raw/Times.csv"
RUTA_MONITORING = "../data/raw/Monitoring.csv"

# En Colab con Drive, por ejemplo:
# from google.colab import drive; drive.mount("/content/drive")
# RUTA_TIMES = "/content/drive/MyDrive/TA_ML/Times.csv"
# RUTA_MONITORING = "/content/drive/MyDrive/TA_ML/Monitoring.csv"

# Si las columnas reales tienen otros nombres, mapearlos aquí: {"nombre_real": "nombre_esperado"}
RENOMBRAR_TIMES = {}
RENOMBRAR_MONITORING = {}

# Si las fechas vienen como 01/03/2024 y significan 1 de marzo, poner True
FECHA_DIA_PRIMERO = False

DIR_SALIDA = Path("../results/auditoria")
DIR_FIG = DIR_SALIDA / "figuras"

# Caja geográfica amplia y aproximada alrededor de Cusco. No es un límite oficial: ajustar si hace falta.
CAJA_CUSCO = {"lat_min": -13.70, "lat_max": -13.35, "lon_min": -72.15, "lon_max": -71.70}

MAX_PUNTOS_FIGURA = 200_000   # muestra máxima de puntos para gráficos
DT_MAX_PAR = 120              # s; pares de puntos GPS más espaciados no se usan para verificar speed ni bearing
DIST_MIN_RUMBO = 15           # m; desplazamiento mínimo para comparar bearing con el rumbo calculado
HUECO_GPS = 300               # s; separación entre puntos que se considera hueco de muestreo
TOLERANCIA_S = 60             # s; tolerancia al comparar horas entre archivos
CELDA_GRADOS = 0.002          # ~220 m; celdas para comparar trazados entre vehículos
MAX_VEH_TRAZADO = 40          # vehículos máximos en la matriz de similitud de trazados
SEMILLA = 42

# --- parámetros nuevos de la v2 ---
LAP_FUERA_DE_OPERACION = 0    # apuntes de clase: lap = 0 es vehículo fuera de operación
UMBRAL_EXTREMO_MIN = 60       # min; |difference_time| mayor a esto se trata como error de registro
UMBRAL_ATRASO_MIN = 5         # min; "atrasado" si difference_time >= este valor
UMBRAL_ADELANTO_MIN = 3       # min; "adelantado" si difference_time <= -este valor
FRACCION_TEST = 0.25          # fracción final de días reservada como prueba en la línea base de referencia

In [2]:
import csv
import json
import shutil
import traceback
import warnings
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")
DIR_FIG.mkdir(parents=True, exist_ok=True)
print("pandas", pd.__version__, "| numpy", np.__version__)

# ---------- registro de hallazgos ----------
HALLAZGOS = []
FIGURAS = []
_SECCION = ["general"]


def _limpio(v):
    if isinstance(v, dict):
        return {str(k): _limpio(x) for k, x in v.items()}
    if isinstance(v, (list, tuple, set)):
        return [_limpio(x) for x in v]
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (float, np.floating)):
        return None if not np.isfinite(v) else round(float(v), 4)
    if v is pd.NaT or v is None:
        return None
    if isinstance(v, (pd.Timestamp, pd.Timedelta)):
        return str(v)
    return v


def registrar(clave, valor, etiqueta="observado"):
    HALLAZGOS.append({"seccion": _SECCION[0], "clave": clave, "valor": _limpio(valor), "etiqueta": etiqueta})
    print(f"[{etiqueta}] {clave}: {_limpio(valor)}")


@contextmanager
def bloque(nombre):
    """Ejecuta un bloque de análisis; si falla, registra el error y deja seguir al resto del notebook."""
    _SECCION[0] = nombre
    print(f"\n========== {nombre} ==========")
    try:
        yield
    except Exception as e:
        registrar("error", f"{type(e).__name__}: {e}", "error")
        traceback.print_exc()
        plt.close("all")


def guardar_fig(nombre, fig=None):
    fig = fig or plt.gcf()
    ruta = DIR_FIG / f"{len(FIGURAS) + 1:02d}_{nombre}.png"
    fig.tight_layout()
    fig.savefig(ruta, dpi=120, bbox_inches="tight")
    FIGURAS.append(ruta.name)
    plt.show()
    plt.close(fig)
    print("Figura guardada:", ruta)


def pct(mascara):
    mascara = pd.Series(mascara)
    return round(100 * float(mascara.mean()), 3) if len(mascara) else float("nan")


def describe_dict(serie, factor=1.0):
    s = pd.to_numeric(pd.Series(serie), errors="coerce").dropna() * factor
    if s.empty:
        return {}
    q = s.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    return {"n": len(s), "media": s.mean(), "min": s.min(), "p01": q[0.01], "p05": q[0.05], "p25": q[0.25],
            "mediana": q[0.5], "p75": q[0.75], "p95": q[0.95], "p99": q[0.99], "max": s.max()}


def recortar(serie, inferior=0.005, superior=0.995):
    s = pd.Series(serie).dropna()
    if s.empty:
        return s
    a, b = s.quantile([inferior, superior])
    return s[(s >= a) & (s <= b)]


# ---------- lectura ----------
COLUMNAS_TEXTO = {"date", "time", "start_time", "end_time", "difference_time"}


def detectar_separador(ruta):
    with open(ruta, "r", encoding="utf-8-sig", errors="replace") as f:
        muestra = f.read(20000)
    try:
        return csv.Sniffer().sniff(muestra, delimiters=",;\t|").delimiter
    except csv.Error:
        return ","


def cargar(ruta, renombrar):
    sep = detectar_separador(ruta)
    cabecera = pd.read_csv(ruta, sep=sep, nrows=0, encoding="utf-8-sig").columns
    dtype = {c: str for c in cabecera if renombrar.get(c.strip(), c.strip()) in COLUMNAS_TEXTO}
    df = pd.read_csv(ruta, sep=sep, dtype=dtype, encoding="utf-8-sig", low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns=renombrar)
    sin_nombre = [c for c in df.columns if c.startswith("Unnamed")]
    if sin_nombre:
        print("Columnas sin nombre eliminadas (índices exportados):", sin_nombre)
        df = df.drop(columns=sin_nombre)
    print(f"{ruta}: separador {sep!r}, {len(df):,} filas, {df.shape[1]} columnas")
    return df, sep


# ---------- conversión de tipos ----------
def normalizar_id(s):
    """Ids como texto comparable entre archivos (12, 12.0 y '12' quedan iguales)."""
    num = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() and num.notna().sum() >= 0.99 * s.notna().sum():
        return num.round().astype("Int64").astype(str).where(num.notna(), None)
    return s.astype(str).str.strip().where(s.notna(), None)


def _por_unicos(s, funcion):
    """Convierte solo los valores únicos y luego mapea: mucho más rápido en archivos grandes."""
    valores = s.dropna().astype(str).str.strip()
    unicos = pd.Series(valores.unique())
    if unicos.empty:
        return pd.Series(np.nan, index=s.index)
    convertidos = funcion(unicos)
    mapa = pd.Series(np.asarray(convertidos), index=unicos.values)
    return valores.map(mapa).reindex(s.index)


def _a_datetime(u):
    r = pd.to_datetime(u, errors="coerce", format="%Y-%m-%d")
    if r.isna().mean() > 0.05:
        r2 = pd.to_datetime(u, errors="coerce", format="mixed", dayfirst=FECHA_DIA_PRIMERO)
        if getattr(r2.dt, "tz", None) is not None:
            r2 = r2.dt.tz_localize(None)
        r = r.fillna(r2)
    return r.astype("datetime64[ns]")


def fechas(s):
    return _por_unicos(s, _a_datetime).astype("datetime64[ns]")


def _horas_a_segundos(u):
    u = u.astype(str).str.strip()
    con_fecha = u.str.match(r"^\d{4}-\d{2}-\d{2}[ T]")
    solo_hora = u.where(~con_fecha, u.str.slice(11))
    con_dos_puntos = solo_hora.where(solo_hora.str.contains(":", na=False))
    seg = pd.to_timedelta(con_dos_puntos, errors="coerce").dt.total_seconds()
    falla = seg.isna()
    if falla.any():
        dt = pd.to_datetime(solo_hora[falla], errors="coerce", format="mixed")
        seg.loc[falla] = (dt - dt.dt.normalize()).dt.total_seconds()
    return seg


def tiene_fecha(serie):
    m = serie.dropna().astype(str).str.strip().head(2000)
    return len(m) > 0 and m.str.match(r"^\d{4}-\d{2}-\d{2}[ T]\d").mean() > 0.5


def marca_tiempo(df, col):
    """Devuelve (marca de tiempo completa, si la columna ya traía fecha)."""
    if tiene_fecha(df[col]):
        return fechas(df[col]), True
    seg = _por_unicos(df[col], _horas_a_segundos)
    return (df["fecha"] + pd.to_timedelta(seg, unit="s")).astype("datetime64[ns]"), False


def duracion_registrada(s):
    """difference_time en número (unidad por determinar) o en segundos si viene como texto de duración."""
    num = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() and num.notna().sum() >= 0.9 * s.notna().sum():
        return num.astype(float), "numerico"
    def conv(u):
        u = u.where(u.str.contains(r"[:a-zA-Z]", na=False))
        return pd.to_timedelta(u, errors="coerce").dt.total_seconds()
    return _por_unicos(s, conv).astype(float), "texto_duracion"


def patron(s):
    return s.astype(str).str.strip().str.replace(r"\d", "9", regex=True).str.replace(r"[A-Za-z]", "a", regex=True)


def tabla_patrones(serie, n=6):
    m = serie.dropna()
    if m.empty:
        return pd.DataFrame(columns=["proporcion", "ejemplo"])
    if len(m) > 300_000:
        m = m.sample(300_000, random_state=SEMILLA)
    p = patron(m)
    conteo = p.value_counts(normalize=True).head(n)
    ejemplos = {pat: m[p == pat].iloc[0] for pat in conteo.index}
    return pd.DataFrame({"proporcion": conteo.round(4), "ejemplo": pd.Series(ejemplos)})


# ---------- geometría ----------
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def rumbo_grados(lat1, lon1, lat2, lon2):
    """Rumbo inicial de 1 a 2: 0° = norte, sentido horario."""
    p1, p2, dl = np.radians(lat1), np.radians(lat2), np.radians(lon2 - lon1)
    x = np.sin(dl) * np.cos(p2)
    y = np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360


def dif_circular(a, b):
    return np.abs(((a - b + 180) % 360) - 180)


def aspecto_mapa(lat):
    return 1 / np.cos(np.radians(float(np.nanmedian(lat))))


DIAS = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]


# ---------- listas por punto de control (formato {a,b,c}) ----------
def es_lista(serie):
    m = serie.dropna().astype(str).str.strip().head(2000)
    return len(m) > 0 and m.str.startswith("{").mean() > 0.5


def hms_a_seg(x):
    try:
        h, m, s = x.split(":")[:3]
        return int(h) * 3600 + int(m) * 60 + int(float(s))
    except Exception:
        return np.nan


def entero_o_nan(x):
    try:
        return float(x)
    except Exception:
        return np.nan


def parsear_lista(serie, convertir):
    """Convierte celdas '{a,b,c}' en una matriz (filas x elementos). Lo no convertible (por ejemplo '-') queda en NaN.
    Las filas cuyo largo difiere del más frecuente quedan en NaN y se cuentan aparte."""
    partes = serie.astype(str).str.strip().str.strip("{}[]() ").str.split(",")
    largos = partes.str.len()
    n = int(largos.mode().iloc[0])
    out = np.full((len(serie), n), np.nan)
    for i, p in enumerate(partes.values):
        if len(p) == n:
            for j, x in enumerate(p):
                out[i, j] = convertir(x.strip().strip('"'))
    return out, largos.to_numpy()

pandas 2.2.3 | numpy 2.4.6


## 1. Carga y estructura (V1)
¿Cuántas filas y columnas tiene cada archivo? ¿Los nombres coinciden con la descripción de los docentes? ¿Qué tipos infiere pandas?

In [5]:
def asegurar_archivos():
    faltan = [r for r in (RUTA_TIMES, RUTA_MONITORING) if not Path(r).exists()]
    if faltan:
        try:
            from google.colab import files
            print("No se encontraron", faltan, ". Súbelos ahora:")
            files.upload()
        except ImportError:
            pass
    faltan = [r for r in (RUTA_TIMES, RUTA_MONITORING) if not Path(r).exists()]
    if faltan:
        raise FileNotFoundError(f"No se encontraron {faltan}. Coloca los CSV junto al notebook o cambia RUTA_TIMES y RUTA_MONITORING.")


asegurar_archivos()
times, SEP_TIMES = cargar(RUTA_TIMES, RENOMBRAR_TIMES)
mon, SEP_MON = cargar(RUTA_MONITORING, RENOMBRAR_MONITORING)

ESPERADAS = {
    "Times": ["id", "lap", "date", "time", "start_time", "end_time", "difference_time", "vehicle_id_id"],
    "Monitoring": ["id", "date", "time", "lap", "latitude", "longitude", "speed", "bearing", "vehicle_id_id"],
}

with bloque("1_estructura"):
    for nombre, df in [("Times", times), ("Monitoring", mon)]:
        faltan = [c for c in ESPERADAS[nombre] if c not in df.columns]
        extra = [c for c in df.columns if c not in ESPERADAS[nombre]]
        registrar(f"{nombre}.filas", len(df))
        registrar(f"{nombre}.columnas", list(df.columns))
        registrar(f"{nombre}.columnas_esperadas_que_faltan", faltan)
        registrar(f"{nombre}.columnas_extra", extra)
        registrar(f"{nombre}.memoria_MB", df.memory_usage(deep=True).sum() / 1e6)
        resumen = pd.DataFrame({
            "tipo_inferido": df.dtypes.astype(str),
            "no_nulos": df.notna().sum(),
            "valores_distintos": df.nunique(),
            "ejemplo": df.iloc[0] if len(df) else None,
        })
        print(f"\n{nombre}")
        display(resumen)
        display(df.head())
        if faltan:
            print(f"ATENCIÓN: faltan columnas en {nombre}. Ajusta RENOMBRAR_{nombre.upper()} y vuelve a ejecutar.")

Times.csv: separador ',', 4,707 filas, 8 columnas
Monitoring.csv: separador ',', 99,559 filas, 9 columnas

========== 1_estructura ==========
[observado] Times.filas: 4707
[observado] Times.columnas: ['id', 'lap', 'date', 'time', 'start_time', 'end_time', 'difference_time', 'vehicle_id_id']
[observado] Times.columnas_esperadas_que_faltan: []
[observado] Times.columnas_extra: []
[observado] Times.memoria_MB: 3.0886

Times


,tipo_inferido,no_nulos,valores_distintos,ejemplo
id,int64,4707,4707,24945
lap,int64,4707,8,1
date,object,4707,27,2025-01-22
time,object,4707,861,05:45:00
start_time,object,4707,1478,"{05:45:00,05:47:00,05:56:00,06:10:00,06:21:00,..."
end_time,object,4707,4679,"{05:44:03,05:44:54,05:53:52,06:09:29,06:16:50,..."
difference_time,object,4707,4606,"{0,-2,-2,0,-4,-4,-4,-3,-1,0,1,1,1,3,2,-1,0,-1}"
vehicle_id_id,int64,4707,45,17


,id,lap,date,time,start_time,end_time,difference_time,vehicle_id_id
0,24945,1,2025-01-22,05:45:00,"{05:45:00,05:47:00,05:56:00,06:10:00,06:21:00,...","{05:44:03,05:44:54,05:53:52,06:09:29,06:16:50,...","{0,-2,-2,0,-4,-4,-4,-3,-1,0,1,1,1,3,2,-1,0,-1}",17
1,6772,1,2025-01-06,06:14:00,"{06:14:00,06:16:00,06:27:00,06:44:00,06:57:00,...","{06:12:55,06:13:46,06:24:00,06:42:25,06:54:09,...","{-1,-2,-3,-1,-2,-5,-5,-5,-5,-3,-1,-1,-2,0,-2,-...",9
2,24976,1,2025-01-22,06:00:00,"{06:00:00,06:02:00,06:11:00,06:25:00,06:36:00,...","{05:58:15,05:59:45,06:09:19,06:25:34,06:34:38,...","{-1,-2,-1,0,-1,-1,-2,0,1,0,0,0,-2,-1,-2,0,0,0}",20
3,22235,2,2025-01-19,09:26:00,"{09:26:00,09:28:00,09:37:00,09:51:00,10:02:00,...","{09:21:09,09:22:23,09:40:05,09:54:22,10:02:14,...","{-4,-5,3,3,0,0,0,0,0,0,1,0,0,0,0,-1,2,4}",44
4,22225,2,2025-01-19,09:14:00,"{09:14:00,09:16:00,09:25:00,09:39:00,09:50:00,...","{09:12:41,09:13:29,09:25:17,09:41:20,09:49:46,...","{-1,-2,0,2,0,2,1,1,1,0,0,-1,-2,-2,-4,-3,2,3}",30


[observado] Monitoring.filas: 99559
[observado] Monitoring.columnas: ['id', 'date', 'time', 'lap', 'latitude', 'longitude', 'speed', 'bearing', 'vehicle_id_id']
[observado] Monitoring.columnas_esperadas_que_faltan: []
[observado] Monitoring.columnas_extra: []
[observado] Monitoring.memoria_MB: 17.1242

Monitoring


,tipo_inferido,no_nulos,valores_distintos,ejemplo
id,int64,99559,99559,602216
date,object,99558,4,2025-01-01
time,object,99558,46549,06:33:38
lap,float64,99558,8,0.0
latitude,float64,99558,83965,-13.551461
longitude,float64,99558,91284,-71.988751
speed,float64,99558,78,24.0
bearing,float64,99558,360,114.0
vehicle_id_id,float64,99558,44,5.0


,id,date,time,lap,latitude,longitude,speed,bearing,vehicle_id_id
0,602216,2025-01-01,06:33:38,0.0,-13.551461,-71.988751,24.0,114.0,5.0
1,606285,2025-01-01,08:02:37,0.0,-13.551343,-71.987746,21.0,53.0,20.0
2,614423,2025-01-01,10:06:37,0.0,-13.526030,-71.983348,18.0,27.0,17.0
3,630699,2025-01-01,14:21:15,0.0,-13.524874,-71.983428,10.0,24.0,40.0
4,638837,2025-01-01,16:42:39,0.0,-13.531148,-71.929771,25.0,105.0,34.0


In [10]:
mon.describe()

,id,lap,latitude,longitude,speed,bearing,vehicle_id_id
count,9.955900e+04,99558.000000,99558.000000,99558.000000,99558.000000,99558.000000,99558.000000
mean,8.430190e+05,2.166556,-13.537469,-71.938757,25.159776,184.065037,23.374144
std,1.727122e+05,1.615532,0.011198,0.044433,11.236878,98.663295,13.214129
min,6.008520e+05,0.000000,-13.582113,-71.991700,0.000000,0.000000,1.000000
25%,6.911705e+05,1.000000,-13.548539,-71.980896,18.000000,104.000000,12.000000
50%,8.102760e+05,2.000000,-13.535675,-71.951714,24.000000,189.000000,25.000000
75%,9.897605e+05,3.000000,-13.526929,-71.896049,32.000000,284.000000,34.000000
max,1.182401e+06,7.000000,-13.518352,-71.810427,81.000000,359.000000,45.000000


## 2. Calidad (V12)
¿Qué columnas tienen faltantes y en qué proporción? ¿Hay duplicados exactos, ids repetidos o filas repetidas si no se cuenta el id?

In [ ]:
NULOS_TEXTO = {"", "null", "none", "nan", "nat", "na", "n/a"}

with bloque("2_calidad"):
    for nombre, df in [("Times", times), ("Monitoring", mon)]:
        vacios = {}
        for c in df.columns:
            if not pd.api.types.is_numeric_dtype(df[c]):
                vacios[c] = int(df[c].dropna().astype(str).str.strip().str.lower().isin(NULOS_TEXTO).sum())
            else:
                vacios[c] = 0
        tabla = pd.DataFrame({"nulos": df.isna().sum(), "texto_vacio_o_nulo": pd.Series(vacios)})
        tabla["pct"] = (100 * tabla.sum(axis=1) / max(len(df), 1)).round(3)
        print(f"\n{nombre}")
        display(tabla)
        registrar(f"{nombre}.pct_faltantes_por_columna", tabla["pct"].to_dict())
        registrar(f"{nombre}.duplicados_exactos", int(df.duplicated().sum()))
        if "id" in df.columns:
            registrar(f"{nombre}.ids_repetidos", int(df["id"].duplicated().sum()))
        sin_id = [c for c in df.columns if c != "id"]
        registrar(f"{nombre}.duplicados_sin_contar_id", int(df.duplicated(subset=sin_id).sum()))

## 3. Formatos temporales (V2, V11, V19)
¿Qué formato tienen date, time, start_time, end_time y difference_time? ¿Hay mezcla de formatos, fracciones de segundo, AM/PM o zona horaria?

In [ ]:
with bloque("3_formatos_temporales"):
    for nombre, df, cols in [("Times", times, ["date", "time", "start_time", "end_time", "difference_time"]),
                             ("Monitoring", mon, ["date", "time"])]:
        for c in cols:
            if c not in df.columns:
                continue
            if es_lista(df[c]):
                n_el = df[c].astype(str).str.count(",") + 1
                registrar(f"{nombre}.{c}.es_lista", True)
                registrar(f"{nombre}.{c}.elementos_por_celda", n_el.value_counts().head(5).to_dict())
                print(f"\n{nombre}.{c}: lista de {int(n_el.mode().iloc[0])} elementos. Ejemplo:", df[c].iloc[0])
                continue
            tp = tabla_patrones(df[c])
            print(f"\n{nombre}.{c}")
            display(tp)
            registrar(f"{nombre}.{c}.patrones", {k: float(v) for k, v in tp["proporcion"].items()})
            if len(tp) > 1 and tp["proporcion"].iloc[0] < 0.99:
                registrar(f"{nombre}.{c}.formato_mixto", True, "indicio")

In [ ]:
with bloque("3b_preparacion_times"):
    for c in ("start_time", "end_time", "difference_time"):
        if not es_lista(times[c]):
            raise ValueError(f"{c} no viene como lista '{{a,b,...}}'. Este notebook espera una lista por punto de control.")
    T = times.copy()
    T["veh"] = normalizar_id(T["vehicle_id_id"])
    T["lap_n"] = pd.to_numeric(T["lap"], errors="coerce")
    T["fecha"] = fechas(T["date"])
    T["ts_reg"], _ = marca_tiempo(T, "time")

    S, largo_s = parsear_lista(T["start_time"], hms_a_seg)        # hora programada en cada punto de control
    E, largo_e = parsear_lista(T["end_time"], hms_a_seg)          # hora real en cada punto de control
    D, largo_d = parsear_lista(T["difference_time"], entero_o_nan)  # diferencia registrada (min)
    NP = S.shape[1]
    registrar("Times.puntos_de_control_por_vuelta", NP)
    registrar("Times.pct_celdas_con_largo_distinto_al_modal", pct((largo_s != NP) | (largo_e != NP) | (largo_d != NP)))

    E_cero = (E == 0)
    registrar("Times.pct_end_time_igual_a_00_00_00", pct(E_cero.ravel()))
    E = np.where(E_cero, np.nan, E)  # 00:00:00 se trata como punto no registrado (hipótesis; ver coincidencia abajo)
    registrar("Times.pct_difference_time_faltante_guion", pct(np.isnan(D).ravel()))
    registrar("Times.pct_puntos_donde_faltan_ambos_o_ninguno", pct((np.isnan(E) == np.isnan(D)).ravel()))
    registrar("Times.pct_vueltas_con_los_18_puntos_completos", pct(~np.isnan(D).any(axis=1)))
    registrar("Times.pct_start_time_en_minutos_exactos", pct((S % 60 == 0).ravel()))

    def a_ts(seg):
        return (T["fecha"] + pd.to_timedelta(seg, unit="s")).astype("datetime64[ns]")

    e_ini = pd.DataFrame(E).bfill(axis=1).iloc[:, 0].to_numpy()   # primer end_time válido
    e_fin = pd.DataFrame(E).ffill(axis=1).iloc[:, -1].to_numpy()  # último end_time válido
    T["ts_ini"] = a_ts(S[:, 0])                                   # inicio programado
    T["ts_fin_prog"] = a_ts(S[:, -1])
    T["ts_fin"] = a_ts(e_fin)                                     # último punto con hora real
    T["v_ini"] = a_ts(np.fmin(S[:, 0], e_ini))                    # ventana de la vuelta: de lo más temprano
    T["v_fin"] = a_ts(np.fmax(S[:, -1], e_fin))                   # a lo más tardío entre programado y real
    T["hora_ini"] = T["ts_ini"].dt.hour
    T["dia_semana"] = T["ts_ini"].dt.dayofweek
    registrar("Times.pct_no_convertible.fecha", pct(T["fecha"].isna()))
    registrar("Times.vueltas_programadas_que_cruzan_medianoche", int((S[:, -1] < S[:, 0]).sum()))

### difference_time frente a end_time menos start_time (V2)
En los datos reales `difference_time` es una lista de enteros, uno por punto de control. La hipótesis es que cada elemento es la diferencia en minutos entre la hora real (`end_time`) y la programada (`start_time`) de ese punto. Se prueba con tres reglas de redondeo y se define una regla provisional para los valores extremos.

In [ ]:
with bloque("4a_difference_time"):
    dif_min = (E - S) / 60
    ok = ~np.isnan(D) & ~np.isnan(E)
    registrar("difference_time.puntos_comparables", int(ok.sum()))
    tasas = {n: pct(f(dif_min[ok]) == D[ok]) for n, f in {"truncado": np.trunc, "redondeado": np.round, "piso": np.floor}.items()}
    registrar("difference_time.pct_coincidencia_segun_redondeo", tasas)
    mejor = max(tasas, key=tasas.get)
    registrar("difference_time.definicion_probable",
              f"minutos enteros ({mejor}) de end_time - start_time en cada punto de control" if tasas[mejor] >= 95 else "no determinada",
              "indicio")
    registrar("difference_time.minutos_todos_los_puntos", describe_dict(D[~np.isnan(D)]))

    EXTREMO = np.abs(D) > UMBRAL_EXTREMO_MIN
    registrar(f"difference_time.puntos_con_valor_absoluto_mayor_a_{UMBRAL_EXTREMO_MIN}min", int(EXTREMO.sum()))
    registrar("difference_time.pct_puntos_extremos", pct(EXTREMO[~np.isnan(D)]))
    registrar("difference_time.extremos_por_punto_de_control", {i + 1: int(v) for i, v in enumerate(EXTREMO.sum(axis=0))})
    registrar("difference_time.pct_vueltas_con_algun_extremo", pct(EXTREMO.any(axis=1)))
    registrar("difference_time.extremos_con_end_time_justo_despues_de_medianoche", int((EXTREMO & (E < 3600)).sum()))
    D_LIMPIA = np.where(EXTREMO, np.nan, D)
    registrar("difference_time.regla_de_limpieza",
              f"|difference_time| > {UMBRAL_EXTREMO_MIN} min se trata como error de registro", "decision_provisional")
    cl = D_LIMPIA[~np.isnan(D_LIMPIA)]
    registrar("difference_time.limpia_minutos", describe_dict(cl))
    registrar("difference_time.limpia_pct_adelantado_a_tiempo_atrasado",
              {"adelantado_(<=-%d)" % UMBRAL_ADELANTO_MIN: pct(cl <= -UMBRAL_ADELANTO_MIN),
               "a_tiempo": pct((cl > -UMBRAL_ADELANTO_MIN) & (cl < UMBRAL_ATRASO_MIN)),
               "atrasado_(>=%d)" % UMBRAL_ATRASO_MIN: pct(cl >= UMBRAL_ATRASO_MIN)})
    registrar("difference_time.desviacion_estandar_por_punto_limpia",
              [round(float(np.nanstd(D_LIMPIA[:, j])), 2) for j in range(NP)])

    datos = pd.DataFrame(D_LIMPIA, columns=range(1, NP + 1)).melt(var_name="punto", value_name="min").dropna()
    fig, ax = plt.subplots(1, 2, figsize=(15, 4))
    sns.boxplot(data=datos, x="punto", y="min", showfliers=False, ax=ax[0], color="#9ecae1")
    ax[0].axhline(0, color="k", lw=0.8)
    ax[0].set_title("Adelanto (-) o atraso (+) por punto de control, sin valores extremos")
    ax[0].set_xlabel("punto de control")
    ax[0].set_ylabel("min")
    sns.histplot(datos["min"], bins=np.arange(-30.5, 40.5, 1), ax=ax[1])
    ax[1].set_title("Distribución de difference_time (limpia, recortada a [-30, 40])")
    ax[1].set_xlabel("min")
    guardar_fig("desviacion_por_punto", fig)

### time frente a start_time y end_time (V11)
¿`time` es la hora programada del primer punto, la del último, o el momento en que se grabó el registro? Si coincide con el cierre de la vuelta revela el resultado y no puede usarse como predictora.

In [ ]:
def envolver(seg):
    return ((seg + 43200) % 86400) - 43200


with bloque("4b_time_vs_start_end"):
    d_ini = envolver((T["ts_reg"] - T["ts_ini"]).dt.total_seconds())
    d_fin = envolver((T["ts_reg"] - T["ts_fin"]).dt.total_seconds())
    registrar("time_menos_start_time_primer_punto_s", describe_dict(d_ini))
    registrar("pct_time_igual_a_start_time_del_primer_punto", pct(d_ini.abs() <= 1))
    registrar("pct_time_a_menos_de_60s_del_ultimo_end_time", pct(d_fin.abs() <= 60))
    if pct(d_ini.abs() <= 1) > 95:
        registrar("time.interpretacion", "time es la hora programada de salida (start_time del primer punto): se conoce de antemano y no filtra el resultado", "indicio")
    elif pct(d_fin.abs() <= 60) > 80:
        registrar("time.interpretacion", "time coincide con el cierre de la vuelta: excluirlo como predictora", "indicio")
    else:
        registrar("time.interpretacion", "time no coincide de forma sistemática con inicio ni fin: revisar", "indicio")

    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    sns.histplot(d_ini.clip(-3600, 3600), bins=80, ax=ax[0])
    ax[0].set_title("time - start_time del primer punto (s)")
    sns.histplot(d_fin.clip(-3600, 3600), bins=80, ax=ax[1])
    ax[1].set_title("time - último end_time válido (s, recortado a ±1 h)")
    guardar_fig("time_vs_start_end", fig)

## 4. Times: granularidad, lap, cobertura y duración (V3, V4, V5, V14, V15, V16)

In [ ]:
CLAVE = ["veh", "fecha", "lap_n"]

with bloque("4c_granularidad_y_lap"):
    filas_por_clave = T.groupby(CLAVE).size()
    registrar("Times.claves_vehiculo_fecha_lap", len(filas_por_clave))
    registrar("Times.pct_claves_con_mas_de_una_fila", pct(filas_por_clave > 1))
    registrar("Times.distribucion_filas_por_clave", filas_por_clave.value_counts().sort_index().head(10).to_dict())
    if pct(filas_por_clave > 1) > 5:
        registrar("Times.granularidad", "varias filas por vuelta: posible nivel de tramo entre controles", "indicio")
    else:
        registrar("Times.granularidad", "una fila por vuelta", "indicio")

    g = T.groupby(["veh", "fecha"])["lap_n"]
    lap_vd = pd.DataFrame({"min": g.min(), "max": g.max(), "distintos": g.nunique(), "filas": g.size()})
    lap_vd["consecutivo"] = (lap_vd["max"] - lap_vd["min"] + 1) == lap_vd["distintos"]
    registrar("lap.valor_minimo_por_vehiculo_dia_frecuencias", lap_vd["min"].value_counts().head(5).to_dict())
    registrar("lap.maximo_por_vehiculo_dia", describe_dict(lap_vd["max"]))
    registrar("lap.pct_vehiculo_dia_con_laps_consecutivos", pct(lap_vd["consecutivo"]))
    reinicia = pct(lap_vd["min"].isin([0, 1]))
    registrar("lap.pct_vehiculo_dia_que_empiezan_en_0_o_1", reinicia)
    registrar("lap.interpretacion",
              "contador diario por vehículo" if reinicia > 80 else "no parece reiniciar cada día: revisar si es global",
              "indicio")

    o = T.dropna(subset=["ts_ini", "lap_n"]).sort_values(["veh", "ts_ini"], kind="mergesort")
    mismo = o["veh"].eq(o["veh"].shift()) & o["fecha"].eq(o["fecha"].shift())
    baja = (o["lap_n"].diff() < 0) & mismo
    registrar("lap.pct_transiciones_donde_lap_disminuye_al_avanzar_start_time",
              round(100 * baja.sum() / max(int(mismo.sum()), 1), 3))
    display(lap_vd.head(10))
    registrar("Times.unidad_de_observacion",
              f"una fila = una vuelta con {NP} puntos de control (listas); {int(np.sum(~np.isnan(D)))} observaciones punto-vuelta con difference_time",
              "indicio")

In [ ]:
with bloque("4d_cobertura_temporal_times"):
    f = T["fecha"].dropna().dt.normalize()
    rango = pd.date_range(f.min(), f.max(), freq="D")
    faltan = rango.difference(pd.DatetimeIndex(f.unique()))
    registrar("Times.fecha_min", str(f.min().date()))
    registrar("Times.fecha_max", str(f.max().date()))
    registrar("Times.dias_con_datos", int(f.nunique()))
    registrar("Times.dias_en_el_rango", len(rango))
    registrar("Times.dias_sin_datos_dentro_del_rango", [str(d.date()) for d in faltan[:30]])
    registrar("Times.vueltas_por_dia_semana",
              {DIAS[int(k)]: int(v) for k, v in T["dia_semana"].value_counts().sort_index().items()})
    registrar("Times.vueltas_por_hora_de_inicio",
              {int(k): int(v) for k, v in T["hora_ini"].value_counts().sort_index().items()})

    fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
    f.value_counts().sort_index().plot(ax=ax[0], marker=".")
    ax[0].set_title("Times: registros por día")
    ax[0].set_xlabel("")
    T["hora_ini"].value_counts().sort_index().plot.bar(ax=ax[1])
    ax[1].set_title("Times: vueltas por hora de inicio")
    guardar_fig("times_cobertura", fig)

In [ ]:
with bloque("4e_duracion_programada_y_real"):
    dur_prog = (S[:, -1] - S[:, 0]) / 60
    dur_real = (E[:, -1] - E[:, 0]) / 60
    T["dur_prog_min"], T["dur_real_min"] = dur_prog, dur_real
    T["duracion_s"] = dur_real * 60
    registrar("duracion_programada_min", describe_dict(dur_prog))
    registrar("pct_vueltas_con_primer_y_ultimo_punto_con_hora_real", pct(~np.isnan(dur_real)))
    registrar("duracion_real_min_primer_a_ultimo_punto", describe_dict(dur_real))
    registrar("duracion_real_menos_programada_min", describe_dict(dur_real - dur_prog))
    registrar("pct_duracion_real_negativa_o_mayor_a_6h", pct((dur_real <= 0) | (dur_real > 360)))
    registrar("horarios_programados_distintos", int(T["start_time"].nunique()))
    registrar("pct_horarios_programados_no_monotonos", pct(~(np.diff(S, axis=1) >= 0).all(axis=1)))
    seg_prog = np.diff(S, axis=1) / 60
    registrar("tiempo_programado_entre_puntos_consecutivos_min_mediana", [round(float(np.median(seg_prog[:, j])), 1) for j in range(NP - 1)])
    registrar("pct_vueltas_con_inicio_programado_antes_de_las_5h", pct(T["hora_ini"] < 5))

    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    sns.histplot(dur_prog, bins=40, ax=ax[0])
    ax[0].set_title("Duración programada (min)")
    v = (dur_real - dur_prog)
    sns.histplot(v[(v > -60) & (v < 60)], bins=60, ax=ax[1])
    ax[1].set_title("Duración real menos programada (min, entre -60 y 60)")
    guardar_fig("times_duracion", fig)

In [ ]:
with bloque("4f_registros_por_vehiculo"):
    por_veh = T.groupby("veh").size().sort_values(ascending=False)
    registrar("Times.vehiculos_distintos", len(por_veh))
    registrar("Times.vueltas_por_vehiculo", describe_dict(por_veh))
    top = max(1, int(np.ceil(0.1 * len(por_veh))))
    registrar("Times.pct_vueltas_del_10pct_de_vehiculos_con_mas_vueltas", round(100 * por_veh.head(top).sum() / por_veh.sum(), 2))
    registrar("Times.vueltas_por_vehiculo_y_dia", describe_dict(T.groupby(["veh", "fecha"]).size()))
    plt.figure(figsize=(12, 3.5))
    por_veh.head(40).plot.bar()
    plt.title("Times: vueltas por vehículo (hasta 40 con más registros)")
    guardar_fig("times_por_vehiculo")

In [ ]:
with bloque("4g_descanso_entre_vueltas"):
    o = T.dropna(subset=["ts_ini", "ts_fin"]).sort_values(["veh", "ts_ini"], kind="mergesort")
    siguiente = o["veh"].eq(o["veh"].shift(-1)) & o["fecha"].eq(o["fecha"].shift(-1))
    descanso = (o["ts_ini"].shift(-1) - o["ts_fin"]).dt.total_seconds().where(siguiente).dropna()
    registrar("descanso_entre_vueltas_min (inicio programado siguiente menos ultimo end_time)", describe_dict(descanso, 1 / 60))
    registrar("pct_vueltas_que_se_solapan_con_la_siguiente", pct(descanso < 0))
    plt.figure(figsize=(7, 3.5))
    sns.histplot(recortar(descanso / 60, 0.0, 0.99), bins=60)
    plt.title("Inicio programado de la vuelta siguiente menos fin real de la actual")
    plt.xlabel("min")
    guardar_fig("times_descanso")

## 4bis. Puntualidad: dependencia entre puntos y línea base de referencia
Estas dos celdas no existían en la v1. Miden qué tanto del atraso final se puede anticipar con el atraso ya observado. Sirven para justificar el problema del plan v2; no son experimentos.

In [ ]:
with bloque("4h_puntualidad_y_dependencia"):
    DL = pd.DataFrame(D_LIMPIA, columns=range(1, NP + 1))
    corr = DL.corr(min_periods=100)
    adj = [corr.iloc[i, i + 1] for i in range(NP - 1)]
    registrar("puntualidad.correlacion_entre_puntos_consecutivos_media", float(np.nanmean(adj)))
    registrar("puntualidad.correlacion_de_cada_punto_con_el_ultimo", [round(float(x), 3) for x in corr.iloc[:, -1].values])
    registrar("puntualidad.pct_atrasado_por_punto", [round(pct(DL[c].dropna() >= UMBRAL_ATRASO_MIN), 2) for c in DL.columns])
    registrar("puntualidad.pct_adelantado_por_punto", [round(pct(DL[c].dropna() <= -UMBRAL_ADELANTO_MIN), 2) for c in DL.columns])
    T["d_media"] = DL.mean(axis=1).to_numpy()
    registrar("puntualidad.media_por_dia_de_semana_min", {DIAS[int(k)]: round(float(v), 2) for k, v in T.groupby("dia_semana")["d_media"].mean().items()})
    registrar("puntualidad.media_por_hora_de_inicio_min", {int(k): round(float(v), 2) for k, v in T.groupby("hora_ini")["d_media"].mean().items()})
    registrar("puntualidad.vueltas_por_vehiculo_media_atraso_p05_p95",
              {"p05": float(T.groupby("veh")["d_media"].mean().quantile(0.05)), "p95": float(T.groupby("veh")["d_media"].mean().quantile(0.95))})

    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(corr, cmap="vlag", center=0, vmin=-0.2, vmax=1, ax=ax[0], square=True, cbar_kws={"shrink": 0.7})
    ax[0].set_title("Correlación entre puntos de control (difference_time limpia)")
    sns.barplot(x=T["hora_ini"], y=T["d_media"], ax=ax[1], color="#9ecae1", errorbar=None)
    ax[1].axhline(0, color="k", lw=0.8)
    ax[1].set_title("Desviación media de la vuelta por hora de inicio programada (min)")
    ax[1].set_xlabel("hora de inicio")
    guardar_fig("puntualidad_dependencia", fig)

In [ ]:
with bloque("4i_linea_base_referencia"):
    dias = np.sort(T["fecha"].dropna().unique())
    corte = dias[int(np.floor(len(dias) * (1 - FRACCION_TEST)))]
    test = (T["fecha"] >= corte).to_numpy()
    train = ~test
    registrar("referencia.vueltas_entrenamiento_y_prueba", {"entrenamiento": int(train.sum()), "prueba": int(test.sum())}, "referencia_preliminar")
    registrar("referencia.prueba_desde", str(pd.Timestamp(corte).date()), "referencia_preliminar")
    destino = NP - 1
    filas = []
    for k in range(1, NP - 1):
        mtr = train & ~np.isnan(D_LIMPIA[:, k]) & ~np.isnan(D_LIMPIA[:, destino])
        mte = test & ~np.isnan(D_LIMPIA[:, k]) & ~np.isnan(D_LIMPIA[:, destino])
        if mtr.sum() < 30 or mte.sum() < 30:
            continue
        x_tr, y_tr = D_LIMPIA[mtr, k], D_LIMPIA[mtr, destino]
        x_te, y_te = D_LIMPIA[mte, k], D_LIMPIA[mte, destino]
        b = np.polyfit(x_tr, y_tr, 1)
        filas.append({"ultimo_punto_observado": k + 1, "n_prueba": int(mte.sum()),
                      "MAE_cero": np.abs(y_te).mean(), "MAE_mediana_entrenamiento": np.abs(y_te - np.median(y_tr)).mean(),
                      "MAE_persistencia": np.abs(y_te - x_te).mean(), "MAE_lineal": np.abs(y_te - np.polyval(b, x_te)).mean()})
    tabla = pd.DataFrame(filas).round(3)
    display(tabla)
    registrar("referencia.MAE_min_al_predecir_el_ultimo_punto_segun_el_punto_observado", tabla.to_dict("records"), "referencia_preliminar")
    registrar("referencia.nota", "Predecir difference_time del punto 18 con el del punto k, partición temporal por días, sin extremos. Es un contexto para el plan, no un experimento.", "referencia_preliminar")
    plt.figure(figsize=(8, 4))
    for c in ["MAE_cero", "MAE_persistencia", "MAE_lineal"]:
        plt.plot(tabla["ultimo_punto_observado"], tabla[c], marker="o", label=c)
    plt.xlabel("último punto de control observado")
    plt.ylabel("MAE sobre el punto 18 (min)")
    plt.title("Referencia: qué tanto ayuda conocer el atraso parcial")
    plt.legend()
    guardar_fig("linea_base_referencia")

## 5. Monitoring: muestreo, coordenadas, speed, bearing y trazados (V7 a V10, V13, V15, V18)

In [ ]:
with bloque("5a_preparacion_monitoring"):
    M = mon.copy()
    M["veh"] = normalizar_id(M["vehicle_id_id"])
    M["lap_n"] = pd.to_numeric(M["lap"], errors="coerce")
    M["fecha"] = fechas(M["date"])
    M["ts"], _ = marca_tiempo(M, "time")
    for c in ["latitude", "longitude", "speed", "bearing"]:
        M[c] = pd.to_numeric(M[c], errors="coerce")
    registrar("Monitoring.pct_no_convertible.fecha", pct(M["fecha"].isna()))
    registrar("Monitoring.pct_no_convertible.ts", pct(M["ts"].isna()))

    M = M.sort_values(["veh", "ts"], kind="mergesort").reset_index(drop=True)
    M["mismo_tramo"] = M["veh"].eq(M["veh"].shift()) & M["fecha"].eq(M["fecha"].shift())
    lat0, lon0 = M["latitude"].shift(), M["longitude"].shift()
    M["dt_s"] = M["ts"].diff().dt.total_seconds().where(M["mismo_tramo"])
    M["dist_m"] = haversine_m(lat0, lon0, M["latitude"], M["longitude"]).where(M["mismo_tramo"])
    M["rumbo_calc"] = rumbo_grados(lat0, lon0, M["latitude"], M["longitude"]).where(M["mismo_tramo"])
    M["v_calc_ms"] = (M["dist_m"] / M["dt_s"]).where(M["dt_s"] > 0)
    M["hora"] = M["ts"].dt.hour
    M["en_operacion"] = M["lap_n"].fillna(LAP_FUERA_DE_OPERACION) != LAP_FUERA_DE_OPERACION

In [ ]:
with bloque("5b_coordenadas"):
    lat, lon = M["latitude"], M["longitude"]
    invalida = lat.isna() | lon.isna() | (lat.abs() > 90) | (lon.abs() > 180) | ((lat == 0) & (lon == 0))
    en_caja = lat.between(CAJA_CUSCO["lat_min"], CAJA_CUSCO["lat_max"]) & lon.between(CAJA_CUSCO["lon_min"], CAJA_CUSCO["lon_max"])
    registrar("coord.pct_invalidas_o_0_0", pct(invalida))
    registrar("coord.pct_dentro_de_caja_aproximada_de_Cusco", pct(en_caja[~invalida]))
    registrar("coord.latitud", describe_dict(lat[~invalida]))
    registrar("coord.longitud", describe_dict(lon[~invalida]))
    if lat[~invalida].median() < -60 and lon[~invalida].median() > -20:
        registrar("coord.posible_intercambio_lat_lon", True, "indicio")
    if pct(en_caja[~invalida]) > 50:
        M["coord_ok"] = ~invalida & en_caja
    else:
        M["coord_ok"] = ~invalida
        registrar("coord.caja_no_aplicada", "la mayoría cae fuera de la caja: revisar CAJA_CUSCO", "indicio")
    registrar("coord.pct_pares_con_velocidad_calculada_mayor_a_144_kmh", pct(M["v_calc_ms"].dropna() > 40))

In [ ]:
with bloque("5c_muestreo_y_cobertura"):
    dt = M["dt_s"].dropna()
    registrar("muestreo.dt_entre_puntos_s", describe_dict(dt))
    registrar("muestreo.pct_dt_cero_marca_repetida", pct(dt == 0))
    registrar(f"muestreo.pct_huecos_mayores_a_{HUECO_GPS}s", pct(dt > HUECO_GPS))
    registrar("Monitoring.vehiculos_distintos", int(M["veh"].nunique()))
    registrar("Monitoring.puntos_por_vehiculo_y_dia", describe_dict(M.groupby(["veh", "fecha"]).size()))
    f = M["fecha"].dropna().dt.normalize()
    rango = pd.date_range(f.min(), f.max(), freq="D")
    registrar("Monitoring.fecha_min", str(f.min().date()))
    registrar("Monitoring.fecha_max", str(f.max().date()))
    registrar("Monitoring.dias_con_datos", int(f.nunique()))
    registrar("Monitoring.dias_sin_datos_dentro_del_rango",
              [str(d.date()) for d in rango.difference(pd.DatetimeIndex(f.unique()))[:30]])
    registrar("Monitoring.puntos_por_hora", {int(k): int(v) for k, v in M["hora"].value_counts().sort_index().items()})

    fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
    sns.histplot(dt[dt <= dt.quantile(0.99)], bins=60, ax=ax[0])
    ax[0].set_title("Segundos entre puntos GPS consecutivos (hasta p99)")
    f.value_counts().sort_index().plot(ax=ax[1], marker=".")
    ax[1].set_title("Monitoring: puntos por día")
    ax[1].set_xlabel("")
    guardar_fig("monitoring_muestreo", fig)

In [ ]:
with bloque("5d_lap_en_monitoring"):
    registrar("Monitoring.lap.interpretacion", "lap = 0 es vehículo fuera de operación; lap > 0 es el número de vuelta (apuntes de clase)", "dato_de_clase")
    registrar("Monitoring.lap.pct_nulo", pct(M["lap_n"].isna()))
    registrar("Monitoring.lap.pct_cero_fuera_de_operacion", pct(~M["en_operacion"]))
    registrar("Monitoring.pct_fuera_de_operacion_por_hora", {int(k): round(float(v), 2) for k, v in (100 * (~M["en_operacion"]).groupby(M["hora"]).mean()).items()})
    registrar("Monitoring.laps_por_vehiculo_y_dia_en_operacion", describe_dict(M[M["en_operacion"]].groupby(["veh", "fecha"])["lap_n"].nunique()))
    baja = (M["lap_n"].diff() < 0) & M["mismo_tramo"] & M["en_operacion"] & M["en_operacion"].shift(fill_value=False)
    registrar("Monitoring.lap.pct_transiciones_donde_lap_disminuye_en_el_tiempo_en_operacion",
              round(100 * baja.sum() / max(int((M["mismo_tramo"] & M["en_operacion"]).sum()), 1), 3))
    registrar("Monitoring.puntos_por_vuelta_en_operacion", describe_dict(M[M["en_operacion"]].groupby(CLAVE).size()))

### Unidad de speed (V9)
Se compara speed (del punto de llegada, del de salida o el promedio de ambos, según cuál sea más estable) con la velocidad calculada como distancia haversine entre tiempo, en m/s. Un cociente cercano a 1 indica m/s y uno cercano a 3,6 indica km/h. El ruido GPS hace que solo la mediana sea informativa.

In [ ]:
UNIDAD_SPEED = None

with bloque("5e_unidad_de_speed"):
    s = M["speed"]
    registrar("speed", describe_dict(s))
    registrar("speed.pct_cero", pct(s == 0))
    registrar("speed.pct_negativa", pct(s < 0))
    par = M["dt_s"].between(1, DT_MAX_PAR) & M["coord_ok"] & M["coord_ok"].shift(fill_value=False)
    base_ok = par & (M["v_calc_ms"] > 1) & (M["v_calc_ms"] < 40)
    # speed es instantánea: se prueba con el punto de llegada, el de salida y el promedio de ambos,
    # y se usa la versión cuyo cociente tiene menor dispersión (rango intercuartílico en escala log).
    versiones = {"punto_de_llegada": M["speed"], "punto_de_salida": M["speed"].shift(),
                 "promedio_de_ambos": (M["speed"] + M["speed"].shift()) / 2}
    disp, medianas = {}, {}
    for nombre_v, v in versiones.items():
        okv = base_ok & (v > 0)
        if okv.sum() >= 30:
            lr = np.log(v[okv] / M["v_calc_ms"][okv])
            disp[nombre_v] = float(lr.quantile(0.75) - lr.quantile(0.25))
            medianas[nombre_v] = float(np.exp(lr.median()))
    registrar("speed.cociente_mediano_por_version", medianas)
    registrar("speed.dispersion_log_por_version", disp)
    elegida = min(disp, key=disp.get) if disp else "promedio_de_ambos"
    v_rep = versiones[elegida]
    ok = base_ok & (v_rep > 0)
    cociente = (v_rep[ok] / M["v_calc_ms"][ok])
    registrar("speed.pares_usados", int(ok.sum()))
    if ok.sum() >= 30:
        med = float(cociente.median())
        registrar("speed.version_usada", elegida, "decision_provisional")
        registrar("speed.cociente_reportada_sobre_calculada_en_ms", describe_dict(cociente))
        candidatos = {"m/s": 1.0, "km/h": 3.6, "mph": 2.237, "nudos": 1.944}
        cercano = min(candidatos, key=lambda k: abs(np.log(med / candidatos[k])))
        if abs(np.log(med / candidatos[cercano])) < np.log(1.25):
            UNIDAD_SPEED = cercano
        registrar("speed.unidad_probable", UNIDAD_SPEED or f"no concluyente (cociente mediano {med:.2f})", "indicio")
        muestra = pd.DataFrame({"calc_kmh": M["v_calc_ms"][ok] * 3.6, "rep": v_rep[ok]})
        muestra = muestra.sample(min(len(muestra), 20_000), random_state=SEMILLA)
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].scatter(muestra["calc_kmh"], muestra["rep"], s=2, alpha=0.3)
        x = np.linspace(0, muestra["calc_kmh"].quantile(0.99), 50)
        ax[0].plot(x, x, "r--", label="si speed está en km/h")
        ax[0].plot(x, x / 3.6, "g--", label="si speed está en m/s")
        ax[0].set_xlabel("velocidad calculada (km/h)")
        ax[0].set_ylabel("speed reportada")
        ax[0].legend()
        ax[0].set_title("speed reportada frente a calculada")
        sns.histplot(recortar(s[s > 0]), bins=60, ax=ax[1])
        ax[1].set_title("Distribución de speed (> 0)")
        guardar_fig("monitoring_speed", fig)
    else:
        registrar("speed.unidad_probable", "no concluyente: pocos pares válidos", "indicio")

### Convención de bearing (V10)
Se compara bearing con el rumbo calculado entre puntos consecutivos con desplazamiento real. Si la diferencia mediana es pequeña, bearing usa 0° al norte y sentido horario. También se prueba la convención matemática (0° al este, antihorario).

In [ ]:
with bloque("5f_bearing"):
    b = M["bearing"]
    registrar("bearing", describe_dict(b))
    registrar("bearing.pct_fuera_de_0_360", pct((b < 0) | (b > 360)))
    registrar("bearing.pct_exactamente_0", pct(b == 0))
    mov = (M["dist_m"] >= DIST_MIN_RUMBO) & M["dt_s"].between(1, DT_MAX_PAR) & M["coord_ok"] & M["coord_ok"].shift(fill_value=False)
    calc = M["rumbo_calc"][mov]
    d_actual = dif_circular(b[mov], calc)
    d_previo = dif_circular(b.shift()[mov], calc)
    d_mat = dif_circular(((90 - b) % 360)[mov], calc)
    registrar("bearing.pares_usados", int(mov.sum()))
    registrar("bearing.dif_mediana_con_rumbo_calculado_punto_actual", float(d_actual.median()))
    registrar("bearing.dif_mediana_con_rumbo_calculado_punto_previo", float(d_previo.median()))
    registrar("bearing.dif_mediana_convencion_matematica", float(d_mat.median()))
    registrar("bearing.pct_dif_menor_a_30_grados", pct(np.minimum(d_actual, d_previo) < 30))
    mejor = min(d_actual.median(), d_previo.median())
    if mejor < 30:
        registrar("bearing.interpretacion", "consistente con 0° = norte y sentido horario", "indicio")
    elif d_mat.median() < 30:
        registrar("bearing.interpretacion", "consistente con 0° = este y sentido antihorario", "indicio")
    else:
        registrar("bearing.interpretacion", "no concuerda con el rumbo calculado: recalcular desde coordenadas", "indicio")

    fig = plt.figure(figsize=(12, 4))
    ax1 = fig.add_subplot(1, 2, 1, projection="polar")
    bb = np.radians(b.dropna().sample(min(b.notna().sum(), MAX_PUNTOS_FIGURA), random_state=SEMILLA))
    ax1.hist(bb, bins=36)
    ax1.set_theta_zero_location("N")
    ax1.set_theta_direction(-1)
    ax1.set_title("Distribución de bearing")
    ax2 = fig.add_subplot(1, 2, 2)
    sns.histplot(np.minimum(d_actual, d_previo), bins=36, ax=ax2)
    ax2.set_title("Diferencia entre bearing y rumbo calculado (°)")
    guardar_fig("monitoring_bearing", fig)

In [ ]:
with bloque("5g_detenciones"):
    if UNIDAD_SPEED in ("km/h", "m/s", "mph", "nudos"):
        a_kmh = {"km/h": 1.0, "m/s": 3.6, "mph": 1.609, "nudos": 1.852}[UNIDAD_SPEED]
        detenido = (M["speed"] * a_kmh) < 2
        registrar("detencion.criterio", "speed menor a 2 km/h", "decision_provisional")
    else:
        detenido = M["speed"] == 0
        registrar("detencion.criterio", "speed igual a 0 (unidad no determinada)", "decision_provisional")
    registrar("detencion.pct_puntos_detenidos", pct(detenido))
    registrar("detencion.pct_detenidos_en_operacion", pct(detenido[M["en_operacion"]]))
    registrar("detencion.pct_detenidos_fuera_de_operacion", pct(detenido[~M["en_operacion"]]))
    nueva = detenido.ne(detenido.shift()) | ~M["mismo_tramo"]
    racha = nueva.cumsum()
    R = M[detenido].groupby(racha[detenido]).agg(ini=("ts", "min"), fin=("ts", "max"), n=("ts", "size"),
                                                 lat=("latitude", "median"), lon=("longitude", "median"))
    R["dur_s"] = (R["fin"] - R["ini"]).dt.total_seconds()
    registrar("detencion.rachas_detectadas", len(R))
    registrar("detencion.rachas_de_2_o_mas_puntos", int((R["n"] >= 2).sum()))
    registrar("detencion.duracion_rachas_s", describe_dict(R["dur_s"]))
    registrar("detencion.rachas_de_mas_de_60s", int((R["dur_s"] > 60).sum()))
    registrar("detencion.rachas_de_mas_de_10min", int((R["dur_s"] > 600).sum()))
    DETENCIONES = R

### Trazados por vehículo (V8)
Cada vehículo se resume como el conjunto de celdas de unos 220 m que visita con al menos tres puntos. La similitud de Jaccard entre esos conjuntos indica si todos recorren el mismo trazado (valores altos en toda la matriz) o si hay varias rutas (bloques separados).

In [ ]:
with bloque("5h_trazados_por_vehiculo"):
    P = M[M["coord_ok"] & M["en_operacion"]]
    ilat = np.floor(P["latitude"] / CELDA_GRADOS).astype("int64")
    ilon = np.floor(P["longitude"] / CELDA_GRADOS).astype("int64")
    celdas = pd.DataFrame({"veh": P["veh"].values, "celda": (ilat * 10_000_000 + ilon).values})
    conteo = celdas.value_counts()
    conteo = conteo[conteo >= 3].reset_index()
    vehs = P["veh"].value_counts().index[:MAX_VEH_TRAZADO]
    conjuntos = {v: set(conteo.loc[conteo["veh"] == v, "celda"]) for v in vehs}
    vehs = [v for v in vehs if conjuntos[v]]
    J = pd.DataFrame(np.eye(len(vehs)), index=vehs, columns=vehs)
    for i, a in enumerate(vehs):
        for b_ in vehs[i + 1:]:
            j = len(conjuntos[a] & conjuntos[b_]) / len(conjuntos[a] | conjuntos[b_])
            J.loc[a, b_] = J.loc[b_, a] = j
    fuera = J.values[~np.eye(len(vehs), dtype=bool)]
    registrar("trazado.vehiculos_comparados", len(vehs))
    if len(fuera):
        registrar("trazado.jaccard_entre_vehiculos", describe_dict(fuera))
    try:
        from scipy.cluster.hierarchy import linkage, fcluster
        from scipy.spatial.distance import squareform
        if len(vehs) >= 3:
            Z = linkage(squareform(1 - J.values, checks=False), method="average")
            grupos = fcluster(Z, t=0.7, criterion="distance")
            registrar("trazado.grupos_aproximados_de_trazado", int(len(set(grupos))), "indicio")
            registrar("trazado.vehiculos_por_grupo", pd.Series(grupos).value_counts().to_dict(), "indicio")
            g = sns.clustermap(J, row_linkage=Z, col_linkage=Z, cmap="viridis", vmin=0, vmax=1, figsize=(9, 9))
            g.fig.suptitle("Similitud de Jaccard entre trazados de vehículos", y=1.02)
            guardar_fig("trazados_jaccard", g.fig)
    except ImportError:
        plt.figure(figsize=(8, 7))
        sns.heatmap(J, cmap="viridis", vmin=0, vmax=1)
        plt.title("Similitud de Jaccard entre trazados de vehículos")
        guardar_fig("trazados_jaccard")

In [ ]:
with bloque("5i_mapas"):
    P = M[M["coord_ok"] & M["en_operacion"]]
    muestra = P.sample(min(len(P), MAX_PUNTOS_FIGURA), random_state=SEMILLA)
    asp = aspecto_mapa(muestra["latitude"])
    fig, ax = plt.subplots(1, 2, figsize=(15, 7))
    sc = ax[0].scatter(muestra["longitude"], muestra["latitude"], c=muestra["speed"], s=0.5, cmap="viridis",
                       vmax=muestra["speed"].quantile(0.99), alpha=0.6)
    plt.colorbar(sc, ax=ax[0], label="speed (unidad original)")
    ax[0].set_title("Puntos GPS coloreados por speed")
    for v in P["veh"].value_counts().index[:8]:
        sub = muestra[muestra["veh"] == v]
        ax[1].scatter(sub["longitude"], sub["latitude"], s=0.5, alpha=0.6, label=str(v))
    ax[1].legend(markerscale=12, title="vehículo", fontsize=8)
    ax[1].set_title("Puntos GPS de los 8 vehículos con más registros")
    for a in ax:
        a.set_aspect(asp)
        a.set_xlabel("longitud")
        a.set_ylabel("latitud")
    guardar_fig("mapa_puntos", fig)

## 6. Relación entre Times y Monitoring (V6, V17)
¿Comparten vehículos y fechas? ¿Las claves vehículo, fecha y vuelta coinciden? ¿Los puntos GPS en operación (lap > 0) caen dentro de la ventana de su vuelta? ¿Hay un desfase horario constante? La ventana de una vuelta va de lo más temprano a lo más tardío entre la hora programada y la real.

In [ ]:
DESFASE_H = 0

with bloque("6a_vehiculos_y_fechas"):
    vt, vm = set(T["veh"].dropna()), set(M["veh"].dropna())
    registrar("relacion.vehiculos_times", len(vt))
    registrar("relacion.vehiculos_monitoring", len(vm))
    registrar("relacion.vehiculos_en_ambos", len(vt & vm))
    registrar("relacion.vehiculos_solo_en_times", sorted(vt - vm)[:15])
    registrar("relacion.vehiculos_solo_en_monitoring", sorted(vm - vt)[:15])
    ft, fm = set(T["fecha"].dropna()), set(M["fecha"].dropna())
    registrar("relacion.fechas_en_ambos", len(ft & fm))
    registrar("relacion.fechas_solo_en_times", len(ft - fm))
    registrar("relacion.fechas_solo_en_monitoring", len(fm - ft))

In [ ]:
with bloque("6b_coincidencia_por_clave"):
    Mo = M[M["en_operacion"]]
    kt = T[CLAVE].dropna().drop_duplicates()
    km = Mo[CLAVE].dropna().drop_duplicates()
    comunes = kt.merge(km, on=CLAVE, how="inner")
    registrar("relacion.pct_vueltas_times_con_puntos_en_monitoring", round(100 * len(comunes) / max(len(kt), 1), 3))
    registrar("relacion.pct_vueltas_monitoring_presentes_en_times", round(100 * len(comunes) / max(len(km), 1), 3))
    km_f = km[km["fecha"].isin(set(T["fecha"].dropna()))]
    com_f = kt.merge(km_f, on=CLAVE, how="inner")
    registrar("relacion.pct_vueltas_monitoring_en_fechas_de_times_presentes_en_times", round(100 * len(com_f) / max(len(km_f), 1), 3))

    TV = T.groupby(CLAVE).agg(ini=("ts_ini", "min"), fin=("ts_fin", "max"), v_ini=("v_ini", "min"), v_fin=("v_fin", "max")).reset_index()
    MV = Mo.groupby(CLAVE).agg(m_ini=("ts", "min"), m_fin=("ts", "max"), n_puntos=("ts", "size")).reset_index()
    JV = TV.merge(MV, on=CLAVE, how="inner")
    JV["desfase_ini_s"] = (JV["m_ini"] - JV["ini"]).dt.total_seconds()
    JV["desfase_fin_s"] = (JV["m_fin"] - JV["fin"]).dt.total_seconds()
    registrar("relacion.primer_punto_gps_menos_inicio_programado_s", describe_dict(JV["desfase_ini_s"]))
    registrar("relacion.ultimo_punto_gps_menos_ultimo_end_time_s", describe_dict(JV["desfase_fin_s"]))
    tol5 = pd.Timedelta(minutes=5)
    cubre = (JV["m_ini"] <= JV["v_ini"] + tol5) & (JV["m_fin"] >= JV["v_fin"] - tol5)
    registrar("relacion.pct_vueltas_cuyo_gps_cubre_toda_la_ventana_con_5min_de_tolerancia", pct(cubre))
    registrar("relacion.puntos_gps_por_vuelta_emparejada", describe_dict(JV["n_puntos"]))
    med = JV["desfase_ini_s"].median()
    if med == med and abs(med) >= 1800:
        horas = int(round(med / 3600))
        if horas != 0 and abs(med - horas * 3600) < 900:
            DESFASE_H = horas
            registrar("relacion.posible_desfase_horario_horas", horas, "indicio")
    if len(JV):
        fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
        for a, col, titulo in [(ax[0], "desfase_ini_s", "Primer punto GPS - inicio programado"),
                               (ax[1], "desfase_fin_s", "Último punto GPS - último end_time")]:
            sns.histplot(JV[col].clip(-43200, 43200) / 3600, bins=np.arange(-12, 12.25, 0.25), ax=a)
            a.set_title(f"{titulo} (horas, recortado a ±12 h)")
            a.set_xlabel("horas")
            a.set_ylabel("vueltas")
        guardar_fig("relacion_desfases", fig)

In [ ]:
def cobertura_por_intervalo(desplazamiento_h=0):
    """Asigna cada punto GPS (de cualquier lap) a la vuelta de Times de su vehículo cuya ventana lo contiene, sin usar lap."""
    izq = M.loc[M["ts"].notna() & M["veh"].notna(), ["veh", "ts", "lap_n", "en_operacion"]].copy()
    izq["ts"] = (izq["ts"] - pd.Timedelta(hours=desplazamiento_h)).astype("datetime64[ns]")
    izq = izq.sort_values("ts")
    der = (T.loc[T["v_ini"].notna() & T["veh"].notna(), ["veh", "v_ini", "v_fin", "lap_n"]]
           .rename(columns={"lap_n": "lap_times"}).sort_values("v_ini"))
    asig = pd.merge_asof(izq, der, left_on="ts", right_on="v_ini", by="veh", direction="backward")
    dentro = asig["v_fin"].notna() & (asig["ts"] <= asig["v_fin"] + pd.Timedelta(seconds=TOLERANCIA_S))
    op = asig["en_operacion"]
    concuerda = dentro & op & (asig["lap_n"] == asig["lap_times"])
    return (round(100 * (dentro & op).sum() / max(int(op.sum()), 1), 3),
            round(100 * concuerda.sum() / max(int((dentro & op).sum()), 1), 3),
            round(100 * (dentro & ~op).sum() / max(int((~op).sum()), 1), 3))


with bloque("6c_puntos_dentro_de_su_vuelta"):
    Mo = M[M["en_operacion"]]
    X = Mo[CLAVE + ["ts"]].merge(TV[CLAVE + ["v_ini", "v_fin"]], on=CLAVE, how="left")
    tiene = X["v_ini"].notna()
    tol = pd.Timedelta(seconds=TOLERANCIA_S)
    dentro = tiene & (X["ts"] >= X["v_ini"] - tol) & (X["ts"] <= X["v_fin"] + tol)
    registrar("relacion.pct_puntos_gps_en_operacion_cuya_vuelta_existe_en_times", pct(tiene))
    registrar("relacion.pct_puntos_dentro_de_la_ventana_de_su_vuelta", round(100 * dentro.sum() / max(int(tiene.sum()), 1), 3))
    p_dentro, p_conc, p_lap0 = cobertura_por_intervalo(0)
    registrar("relacion.pct_puntos_en_operacion_dentro_de_alguna_ventana_de_su_vehiculo", p_dentro)
    registrar("relacion.pct_de_esos_puntos_cuyo_lap_coincide_con_el_de_Times", p_conc)
    registrar("relacion.pct_puntos_lap_0_que_caen_dentro_de_una_ventana_de_Times", p_lap0)
    if DESFASE_H:
        a1, a2, _ = cobertura_por_intervalo(DESFASE_H)
        registrar(f"relacion.pct_dentro_corrigiendo_{DESFASE_H}h", a1, "indicio")
        registrar(f"relacion.pct_lap_coincide_corrigiendo_{DESFASE_H}h", a2, "indicio")

In [ ]:
with bloque("6d_inicio_y_fin_de_vueltas"):
    P = M[M["coord_ok"] & M["en_operacion"]]
    g = P.groupby(["veh", "fecha", "lap_n"])
    ini_pts = g[["latitude", "longitude"]].first()
    fin_pts = g[["latitude", "longitude"]].last()
    for nombre, pts in [("inicio", ini_pts), ("fin", fin_pts)]:
        d = haversine_m(pts["latitude"].median(), pts["longitude"].median(), pts["latitude"], pts["longitude"])
        registrar(f"vueltas.distancia_m_de_cada_{nombre}_a_la_mediana_de_{nombre}s", describe_dict(d))
    d_ini_fin = haversine_m(ini_pts["latitude"], ini_pts["longitude"], fin_pts["latitude"], fin_pts["longitude"])
    registrar("vueltas.distancia_m_entre_inicio_y_fin_de_la_misma_vuelta", describe_dict(d_ini_fin))
    if d_ini_fin.median() < 500:
        registrar("vueltas.interpretacion", "las vueltas parecen cerrar el circuito en el mismo lugar", "indicio")

    fondo = P.sample(min(len(P), 50_000), random_state=SEMILLA)
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(fondo["longitude"], fondo["latitude"], s=0.3, c="lightgray")
    ax.scatter(ini_pts["longitude"], ini_pts["latitude"], s=6, c="green", label="primer punto de la vuelta")
    ax.scatter(fin_pts["longitude"], fin_pts["latitude"], s=6, c="red", marker="x", label="último punto de la vuelta")
    ax.set_aspect(aspecto_mapa(fondo["latitude"]))
    ax.legend(markerscale=3)
    ax.set_title("Dónde empiezan y terminan las vueltas")
    guardar_fig("mapa_inicio_fin_vueltas", fig)

## 7. Resumen exportable
Genera salidas_auditoria/resumen_auditoria.md, hallazgos.json y salidas_auditoria.zip. El resumen contiene todos los hallazgos con su etiqueta; sirve de anexo de datos para el informe.

In [ ]:
def _texto(v):
    return v if isinstance(v, str) else json.dumps(v, ensure_ascii=False, default=str)


lineas = [
    "# Resumen de auditoría de datos, TA Aprendizaje de Máquina G5",
    "",
    f"Generado el {pd.Timestamp.now():%Y-%m-%d %H:%M} con pandas {pd.__version__}.",
    f"Archivos: {RUTA_TIMES} (separador {SEP_TIMES!r}) y {RUTA_MONITORING} (separador {SEP_MON!r}).",
    "",
    "Etiquetas: observado es un conteo o estadístico calculado; indicio es una interpretación automática que el equipo debe confirmar; "
    "dato_de_clase viene de los apuntes de clase; decision_provisional es una elección del notebook para seguir; "
    "referencia_preliminar es una cifra de contexto, no un experimento; error indica un bloque que no pudo ejecutarse.",
    "",
]
errores = [h for h in HALLAZGOS if h["etiqueta"] == "error"]
lineas.append(f"Bloques con error: {len(errores)}.")
lineas.append("")
secciones = list(dict.fromkeys(h["seccion"] for h in HALLAZGOS))
for sec in secciones:
    lineas += [f"## {sec}", ""]
    for i, h in enumerate([h for h in HALLAZGOS if h["seccion"] == sec], 1):
        lineas.append(f"{i}. {h['clave']} [{h['etiqueta']}]: {_texto(h['valor'])}")
    lineas.append("")
lineas += ["## Figuras", ""] + [f"{i}. figuras/{n}" for i, n in enumerate(FIGURAS, 1)]

(DIR_SALIDA / "resumen_auditoria.md").write_text("\n".join(lineas), encoding="utf-8")
(DIR_SALIDA / "hallazgos.json").write_text(json.dumps(HALLAZGOS, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
zip_ruta = shutil.make_archive(str(DIR_SALIDA), "zip", DIR_SALIDA)
print(f"Hallazgos: {len(HALLAZGOS)} | figuras: {len(FIGURAS)} | bloques con error: {len(errores)}")
print("Resumen:", DIR_SALIDA / "resumen_auditoria.md")
print("Zip:", zip_ruta)
for h in errores:
    print("  error en", h["seccion"], "->", h["valor"])

try:
    from google.colab import files
    files.download(zip_ruta)
except ImportError:
    pass